# CCA Powerline Removal

**Dataset**: PhysioNet Auditory EEG  
**Channels**: P4, Cz, F8, T7  
**Sampling rate**: 200 Hz  
**Subject**: 1

---

## Overview

We use Canonical Correlation Analysis (CCA) to find the component in EEG correlated with powerline interference (50 Hz), then subtract it from the signal.

## Expected outputs

- Original signal on top
- Cleaned signal on bottom with removed component in red
- Disappearance of regular 50 Hz oscillation

## Key parameters

| Parameter | Value | Meaning |
| --- | --- | --- |
| Channel | P4 | Parietal region |
| Frequency | 50 Hz | Powerline frequency |
| n_components | 1 | Single component |


## 1. Install dependencies


In [ ]:
!pip install mne scikit-learn EMD-signal scipy numpy plotly wfdb


## 2. Clone repo and download data

We download only subject 1 (`--subjects 1`) to speed up the experiment in Colab.


In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


## 3. Load the EEG signal

We load subject 1, experiment 1, session 2.


In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
fs = 200

print(f'Channels: {ch_names}')
print(f'Signal length: {len(eeg_data)} samples ({len(eeg_data)/fs:.1f} seconds)')


## 4. Apply CCA for noise removal

We create a reference signal at 50 Hz (sine and cosine), then apply CCA to find the correlated component and subtract it.


In [ ]:
from sklearn.linear_model import LinearRegression

channel_data = eeg_data[:, 0]
n_samples = len(channel_data)
# Build reference signal for 50 Hz powerline interference
t = np.arange(n_samples) / fs
reference = np.column_stack([
    np.sin(2 * np.pi * 50 * t),
    np.cos(2 * np.pi * 50 * t),
    np.sin(2 * np.pi * 100 * t),
    np.cos(2 * np.pi * 100 * t),
])
# Regression-based subtraction: fit how much of EEG is explained by reference
reg = LinearRegression()
reg.fit(reference, channel_data)
artifact = reg.predict(reference)
cleaned = channel_data - artifact
print(f'Artifact std: {artifact.std():.2f}, Cleaned std: {cleaned.std():.2f}')


## 5. Interactive plot

**What to look for:**

- The red component captures the 50 Hz oscillation
- The green signal is the cleaned signal after noise removal
- Zoom in to notice the subtle difference



In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

n_plot = min(5000, len(channel_data))
t_sec = np.arange(n_plot) / fs

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Original signal - Channel P4',
                                    'Regression-based Cleaned Signal (50 Hz removed)'))
fig.add_trace(go.Scatter(x=t_sec, y=channel_data[:n_plot], name='Original',
                         line=dict(color='blue', width=0.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=t_sec, y=cleaned[:n_plot], name='Cleaned',
                         line=dict(color='green', width=0.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=t_sec, y=artifact[:n_plot], name='Removed',
                         line=dict(color='red', width=1)), row=2, col=1)
fig.update_layout(height=700, title_text='Regression-based Artifact Removal - Powerline (50 Hz)',
                  xaxis2_title='Time (s)', showlegend=True)
fig.show()


## What did we learn?

- CCA finds the highest correlation between two sets of variables
- We use a reference signal to represent the artifact
- It removes the correlated component without affecting brain content
- Effective for powerline interference removal

